<a href="https://colab.research.google.com/github/paolobalasso/DataScienceCourse/blob/main/notebooks/03_demand_eda_multi_product.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 03 · Demand EDA — Multi-Product Hierarchy

**Data Science · Università Cattolica del Sacro Cuore, Milano**
Instructor: *Paolo Balasso*

---

## What you will explore

When demand is a **hierarchy** of products (SKU → Category → Total), three problems appear that do not exist for a single SKU:

1. **Mix problem / Simpson's paradox** — total accuracy can *improve* while every SKU degrades
2. **WAPE vs MAPE on a hierarchy** — same forecast, completely different verdicts
3. **Cannibalisation** — a promo on SKU A boosts A *and* depresses substitute SKU B

This notebook turns the companion deck `DemandMultiProduct.pptx` into runnable code.

**Style.** Short readable code, lots of plots. Mini-exercises are *parameter tweaks* — change category weights to flip the paradox, swap reconciliation methods, inject a cannibalisation event.

---

## 0 · Setup

In [ ]:
%pip install -q matplotlib seaborn pandas numpy

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

NAVY, GOLD, INK, MUTED = '#1B263B', '#D4A017', '#0F141E', '#556070'
GREEN, RED = '#2D6E3E', '#B32D2D'
sns.set_style('whitegrid')
plt.rcParams.update({'font.family': 'DejaVu Sans',
                     'axes.edgecolor': MUTED, 'axes.labelcolor': INK})
np.random.seed(42)
print('Setup complete.')

## 1 · A small multi-SKU hierarchy

We build 6 SKUs grouped into 2 categories:

| Category    | SKU      | Avg weekly units | Notes               |
|-------------|----------|------------------|----------------------|
| Beverages   | BEV_A   | 100              | high volume          |
| Beverages   | BEV_B   | 30               | low volume           |
| Beverages   | BEV_C   | 50               | substitute of BEV_A |
| Snacks      | SNK_A   | 80               | high volume          |
| Snacks      | SNK_B   | 15               | very low volume      |
| Snacks      | SNK_C   | 25               | promo-sensitive      |

The total = Beverages + Snacks = sum of all 6 SKUs. We can forecast at any of three **levels**: Total, Category, SKU.

In [ ]:
def make_hierarchy(n_weeks=104, seed=42):
    rng = np.random.default_rng(seed)
    weeks = pd.date_range('2023-01-02', periods=n_weeks, freq='W-MON')

    config = {
        # SKU       (category, baseline, trend/wk, season_amp, noise_sd)
        'BEV_A':   ('Beverages', 100, 0.20, 12,  6),
        'BEV_B':   ('Beverages',  30, 0.05,  5,  3),
        'BEV_C':   ('Beverages',  50, 0.10,  6,  4),
        'SNK_A':   ('Snacks',    80, 0.15, 10,  5),
        'SNK_B':   ('Snacks',    15, 0.02,  3,  2),
        'SNK_C':   ('Snacks',    25, 0.08,  8,  4),
    }
    data = {}
    cat_of = {}
    t = np.arange(n_weeks)
    for sku, (cat, base, trend, amp, sd) in config.items():
        season = amp*np.sin(2*np.pi*t/52 - 1.2)
        noise  = rng.normal(0, sd, n_weeks)
        data[sku]   = (base + trend*t + season + noise).clip(min=0).round(1)
        cat_of[sku] = cat
    df = pd.DataFrame(data, index=weeks)
    return df, cat_of

df, cat_of = make_hierarchy()
print(f'Shape: {df.shape}  ({df.shape[0]} weeks × {df.shape[1]} SKUs)')
print('\nCategory map:'); print(pd.Series(cat_of))
df.head()

In [ ]:
# Build aggregated views: Category-level and Total
cat_df = (df.T.groupby(pd.Series(cat_of)).sum().T)   # Beverages / Snacks
tot_s  = df.sum(axis=1).rename('TOTAL')
print('Category-level head:'); display(cat_df.head())
print('\nTotal series head:'); display(tot_s.head())

In [ ]:
# Visualise the three hierarchy levels
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
# SKU
for sku in df.columns:
    axes[0].plot(df.index, df[sku], lw=0.9, label=sku, alpha=0.85)
axes[0].set_title('Level 3 — SKU', fontweight='bold', color=INK, loc='left')
axes[0].legend(loc='upper left', ncol=6, fontsize=8)
# Category
for col, color in zip(cat_df.columns, [NAVY, GOLD]):
    axes[1].plot(cat_df.index, cat_df[col], color=color, lw=1.4, label=col)
axes[1].set_title('Level 2 — Category', fontweight='bold', color=INK, loc='left')
axes[1].legend(loc='upper left')
# Total
axes[2].plot(tot_s.index, tot_s.values, color=INK, lw=1.6)
axes[2].set_title('Level 1 — Total', fontweight='bold', color=INK, loc='left')
plt.tight_layout(); plt.show()

## 2 · Naive forecasts at each level

Quick baseline: average of the last 4 weeks. Same logic, three levels.

In [ ]:
H = 12                          # forecast horizon: 12 weeks
train = df.iloc[:-H]; test = df.iloc[-H:]
train_cat, test_cat = cat_df.iloc[:-H], cat_df.iloc[-H:]
train_tot, test_tot = tot_s.iloc[:-H],   tot_s.iloc[-H:]

# Naive: forecast = last 4-week average, repeated for H weeks
def last4_avg(series):
    return np.repeat(series.iloc[-4:].mean(), H)

fc_sku = pd.DataFrame({c: last4_avg(train[c]) for c in train.columns}, index=test.index)
fc_cat = pd.DataFrame({c: last4_avg(train_cat[c]) for c in train_cat.columns}, index=test_cat.index)
fc_tot = pd.Series(last4_avg(train_tot), index=test_tot.index, name='TOTAL')

print('SKU forecasts:'); display(fc_sku.head(3))
print('\nCategory forecasts:'); display(fc_cat.head(3))
print('\nTotal forecast (1st 3 wks):'); print(fc_tot.head(3).round(1).to_string())

## 3 · The mix problem — Simpson's paradox in forecasting

**Claim from the slides:** a total-level forecast can look great while every single SKU is wrong, because errors at SKU level cancel each other.

Below we measure WAPE at each level. The total-level WAPE is often *lower* than the average SKU-level WAPE — exactly the paradox.

In [ ]:
def wape(actual, forecast):
    return np.sum(np.abs(actual - forecast)) / np.sum(np.abs(actual))

# SKU-level WAPE (per SKU and averaged)
sku_wapes = pd.Series({sku: wape(test[sku].values, fc_sku[sku].values) for sku in test.columns})
# Category-level WAPE
cat_wapes = pd.Series({c: wape(test_cat[c].values, fc_cat[c].values) for c in test_cat.columns})
# Total-level WAPE
total_wape = wape(test_tot.values, fc_tot.values)

summary = pd.DataFrame({
    'Level':       ['SKU (avg)', 'SKU (median)', 'Category (avg)', 'Total'],
    'WAPE':        [sku_wapes.mean(), sku_wapes.median(), cat_wapes.mean(), total_wape],
})
summary['WAPE'] = summary['WAPE'].round(4)
print('Per-SKU WAPE:'); print(sku_wapes.round(3).to_string())
print('\nLevel comparison:'); display(summary)
print(f'\nObservation: Total WAPE ({total_wape:.3f}) is typically MUCH lower than the avg SKU WAPE')
print(f'({sku_wapes.mean():.3f}) — errors at SKU level cancel across products. THIS IS THE MIX PROBLEM.')

> **Mini-exercise (parametric).** In `make_hierarchy()`, change the `noise_sd` of BEV_A from 6 to 20. Re-run. Watch SKU-level WAPE explode while Total-level WAPE barely moves — that is the cancellation effect made visible.

## 4 · WAPE vs MAPE on a hierarchy — same forecast, two verdicts

MAPE averages percentage errors; WAPE weights by volume. On a hierarchy with very different SKU volumes (BEV_A=100, SNK_B=15) the two metrics can rank methods differently.

In [ ]:
def mape(actual, forecast):
    a = np.asarray(actual); f = np.asarray(forecast)
    mask = a != 0
    return np.mean(np.abs((a[mask] - f[mask]) / a[mask]))

per_sku = pd.DataFrame({
    'volume':  train.mean(),
    'MAPE':    {sku: mape(test[sku].values, fc_sku[sku].values) for sku in test.columns},
    'WAPE':    {sku: wape(test[sku].values, fc_sku[sku].values) for sku in test.columns},
}).round(3).sort_values('volume', ascending=False)
display(per_sku)

# Aggregate the two metrics in different ways
agg_mape_simple   = per_sku['MAPE'].mean()
agg_wape_simple   = per_sku['WAPE'].mean()
agg_wape_weighted = wape(test.values.flatten(), fc_sku.values.flatten())
print(f'\nSimple-average MAPE across SKUs:    {agg_mape_simple:.3f}')
print(f'Simple-average WAPE across SKUs:    {agg_wape_simple:.3f}')
print(f'Volume-weighted WAPE (pool all):   {agg_wape_weighted:.3f}')

In [ ]:
# Visualise: a small SKU with bad MAPE has tiny impact on volume-weighted WAPE
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(per_sku))
width = 0.35
ax.bar(x - width/2, per_sku['MAPE'], width, label='MAPE', color=GOLD,
       edgecolor='black', linewidth=0.4)
ax.bar(x + width/2, per_sku['WAPE'], width, label='WAPE', color=NAVY,
       edgecolor='black', linewidth=0.4)
ax.set_xticks(x); ax.set_xticklabels(per_sku.index, rotation=0)
ax.set_ylabel('Error')
ax.set_title('MAPE vs WAPE per SKU — same forecast, two scales',
             fontweight='bold', color=INK)
ax.legend(loc='upper right')
plt.tight_layout(); plt.show()
print('\nRule of thumb: use WAPE (volume-weighted) when reporting to business.')
print('MAPE is fine for diagnostics on individual SKUs but never for aggregation.')

## 5 · Reconciliation — bottom-up vs top-down

When you forecast independently at multiple levels, the forecasts will **not add up**. Two classical fixes:

- **Bottom-up.** Forecast each SKU; sum them to get Category and Total. *Pros:* respects SKU structure. *Cons:* noisy SKUs propagate up.
- **Top-down.** Forecast the Total; split it back to SKUs by their historical share. *Pros:* uses the most stable signal. *Cons:* loses per-SKU specifics.

There are also middle-ground methods (MinT, optimal reconciliation) — out of scope for today.

In [ ]:
# Bottom-up: sum SKU forecasts → Total forecast
fc_total_bu = fc_sku.sum(axis=1)
# Top-down: take the Total forecast, split by each SKU's average historical share
share = train.mean() / train.mean().sum()
fc_sku_td = pd.DataFrame(
    np.outer(fc_tot.values, share.values),
    index=fc_tot.index, columns=train.columns,
)
# Re-aggregate to Total to confirm equality
fc_total_td = fc_sku_td.sum(axis=1)
print('Bottom-up Total (1st 3 wks):', fc_total_bu.head(3).round(1).to_list())
print('Top-down  Total (1st 3 wks):', fc_total_td.head(3).round(1).to_list())
print(f'\nActual Total (1st 3 wks):  {test_tot.head(3).round(1).to_list()}')

In [ ]:
# Compare reconciled vs naive baseline on test set
def score(actual, forecast, name):
    return {'method': name, 'WAPE': wape(actual, forecast)}

results = pd.DataFrame([
    score(test_tot.values, fc_tot.values,        'Total naive (last4 avg)'),
    score(test_tot.values, fc_total_bu.values,   'Total bottom-up (sum SKU naive)'),
    score(test_tot.values, fc_total_td.values,   'Total top-down (split Total)'),
]).round(4)
display(results)

# Per-SKU on top-down vs independent SKU naive
per_sku_td = pd.Series(
    {sku: wape(test[sku].values, fc_sku_td[sku].values) for sku in test.columns}
).rename('WAPE_topdown')
per_sku_bu = pd.Series(
    {sku: wape(test[sku].values, fc_sku[sku].values) for sku in test.columns}
).rename('WAPE_bottomup')
print('\nPer-SKU WAPE comparison:')
display(pd.concat([per_sku_bu, per_sku_td], axis=1).round(3))

**Read-out.**
- Top-down is usually **smoother** at Total level (less noise) but **worse** at SKU level (lost specificity).
- Bottom-up is usually **truer to each SKU's behaviour** but **noisier** at Total level (errors don't cancel cleanly).
- The right choice depends on **which level the business cares about** — Total revenue (top-down wins) vs replenishment per SKU (bottom-up wins).

> **Mini-exercise (parametric).** Replace the historical-share split with a **last-4-week share** (more reactive). Re-run. Top-down becomes more sensitive to recent mix shifts — is that better or worse on your holdout?

## 6 · Cannibalisation — when a promo on A hurts B

If BEV_A and BEV_C are substitutes, a promo on BEV_A should *lift* BEV_A and *depress* BEV_C. Independent SKU models ignore this — total category demand might be roughly conserved, but the **mix** shifts.

We inject a 50% promo lift on BEV_A in the last 12 weeks and measure the *substitution effect* on BEV_C.

In [ ]:
# Inject a promo + cannibalisation event
df_cann = df.copy()
promo_start = -12          # last 12 weeks of the series
substitution_rate = 0.40    # 40% of BEV_A's incremental volume cannibalises BEV_C

uplift = df['BEV_A'].iloc[promo_start:] * 0.50   # +50% lift on BEV_A
df_cann.iloc[promo_start:, df_cann.columns.get_loc('BEV_A')] += uplift.values
df_cann.iloc[promo_start:, df_cann.columns.get_loc('BEV_C')] -= substitution_rate * uplift.values
df_cann = df_cann.clip(lower=0)

# Visualise the impact
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
for ax, sku, color in zip(axes, ['BEV_A','BEV_C'], [GREEN, RED]):
    ax.plot(df.index, df[sku],       color=MUTED, lw=0.9, label='No promo', alpha=0.7)
    ax.plot(df_cann.index, df_cann[sku], color=color, lw=1.3, label='With promo')
    ax.axvspan(df.index[promo_start], df.index[-1], color=GOLD, alpha=0.15, label='Promo window')
    ax.set_title(f'{sku}', fontweight='bold', color=INK, loc='left')
    ax.legend(loc='upper left')
plt.suptitle('Cannibalisation: promo on BEV_A → substitution on BEV_C',
             fontweight='bold', color=INK, y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# How much volume moved? Category-level conservation check
bev_pre  = df.iloc[promo_start:][['BEV_A','BEV_B','BEV_C']].sum().sum()
bev_post = df_cann.iloc[promo_start:][['BEV_A','BEV_B','BEV_C']].sum().sum()
print(f'Total Beverage units in last 12 weeks:')
print(f'  Without promo:  {bev_pre:.0f}')
print(f'  With promo:     {bev_post:.0f}')
print(f'  Net incremental: {bev_post - bev_pre:+.0f} units')
print()
print('Interpretation: the promo lifts BEV_A but cannibalises BEV_C.')
print('NET incremental volume = full uplift minus cannibalised volume.')
print('Forecasting BEV_A alone over-states the category gain.')

> **Mini-exercise (parametric).** Try `substitution_rate = 0.0` (no cannibalisation), `0.40` (moderate), `0.95` (near-full substitution). With 0.95 the promo is essentially neutral at category level — just shifts share. Real CPG promos are typically 30–60%.

## 7 · Caveats & next steps

1. **Synthetic data is friendly.** Real retail has zero-demand weeks, intermittent SKUs, holiday spikes, new-product launches → none of these baselines handle them well.
2. **Reconciliation is hot research.** MinT (Wickramasuriya, Athanasopoulos, Hyndman 2019) gives statistically optimal weighted reconciliation — worth knowing.
3. **Cannibalisation needs *causal* analysis**, not just observational. Promo-A and demand-B are co-moving for many reasons (seasonality, weather). Identify substitution with controlled trials or careful synthetic-control methods.
4. **Hierarchy levels carry different KPIs.** Inventory cares about SKU. Revenue cares about Total. Reconciliation choice is a *business* decision, not a *statistical* one.

### Where to go next

- Implement **MinT optimal reconciliation** (Wickramasuriya et al.)
- Promo / price elasticity model on SKU pairs → cross-elasticity matrix
- Hierarchical Bayesian forecasting (PyMC) — shrinkage across SKUs in a category

### References

- Hyndman & Athanasopoulos, *FPP3* — Ch. 11 (Forecasting hierarchical and grouped time series). otexts.com/fpp3
- Wickramasuriya, S., Athanasopoulos, G., Hyndman, R. (2019). *Optimal Forecast Reconciliation for Hierarchical and Grouped Time Series*. JASA.
- Companion deck: `DS_Course_v7_DemandMultiProduct.pptx`.

---

**End of notebook.** ~10 seconds on Colab CPU.